# `sql` — Reference

Execute full ANSI SQL queries directly against DataFrames via Polars' native `SQLContext` engine. The current frame is registered as table **`data`** (and alias **`self`**). Zero external dependencies required—powered 100% by the built-in multithreaded Polars Rust engine.

| Call style | Meaning |
|---|---|
| `pt.sql(df, query)` | Function form; the passed frame is queryable as table `data` (or `self`) |
| `df.sql(query)` | Direct DataFrame method; seamlessly chains with other `pt` methods |
| `df.sql('@query.sql')` | Load and execute SQL query directly from an external `.sql` file |

The names `data` and `self` are reserved for the current frame.
Column names with spaces can use standard SQL double quotes (`"bill length mm"`), square brackets (`[bill length mm]`), or backticks (`` `bill length mm` ``).

---


In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pandas as pd
import pytae as pt

penguins = pt.sample_data['penguins']
tips = pt.sample_data['tips']
titanic = pt.sample_data['titanic']

## Function form — current frame is table `data`

In [2]:
# The passed DataFrame is registered as the table `data`
pt.sql(
    penguins,
    'select species, avg(body_mass_g) as avg_mass, count(*) as n '
    'from data group by species order by species'
)

species,avg_mass,n
str,f64,i64
"""Adelie""",3700.662252,152
"""Chinstrap""",3733.088235,68
"""Gentoo""",5076.01626,124


## DataFrame Method — `df.sql(...)`

Same query engine, but chains naturally off any DataFrame.

In [3]:
# `data` inside the query refers to the frame the accessor is called on
(
    penguins
    .sql("select * from data where species = 'Adelie'")
    .head()
)


species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Adelie""","""Torgersen""",39.1,18.7,181.0,3750.0,"""Male"""
"""Adelie""","""Torgersen""",39.5,17.4,186.0,3800.0,"""Female"""
"""Adelie""","""Torgersen""",40.3,18.0,195.0,3250.0,"""Female"""
"""Adelie""","""Torgersen""",null,null,null,null,null
"""Adelie""","""Torgersen""",36.7,19.3,193.0,3450.0,"""Female"""


## Spaced column names — double quotes, brackets `[col]`, or backticks

SQL standard uses double quotes for identifiers with spaces. `pytae` also accepts SQL-style square brackets `[col a]` and backticks (both automatically converted to standard SQL double quotes before querying):


In [4]:
spaced = pt.DataFrame({'col a': [1, 20, 3], 'col b': ['x', 'y', 'z']})

# Double quotes, brackets, and backticks all work
pt.sql(spaced, 'select [col a], `col b` from data where [col a] > 2')

col a,col b
i64,str
20,"""y"""
3,"""z"""


## Extra frames — joins across DataFrames

Pass additional DataFrames as keyword arguments; each becomes a table of that name.

## Chaining — mix SQL with pandas and other `pt` methods

In [5]:
# SQL to aggregate, then a plain pandas method, then another pt accessor call
(
    tips
    .sql('select day, time, sum(total_bill) as bill, sum(tip) as tip from data group by day, time')
    .sort_values('bill', ascending=False)
    .mutate(tip_pct='tip / bill * 100')
)


day,time,bill,tip,tip_pct
cat,cat,f64,f64,f64
"""Sat""","""Dinner""",1778.4,260.4,14.642375
"""Sun""","""Dinner""",1627.16,247.39,15.203791
"""Thur""","""Lunch""",1077.55,168.83,15.66795
"""Fri""","""Dinner""",235.96,35.28,14.951687
"""Fri""","""Lunch""",89.92,16.68,18.549822
"""Thur""","""Dinner""",18.78,3.0,15.974441


## Loading queries from an external file — `@query.txt`

Just like the CLI (`pytae data.parquet -sql @query.txt`), you can pass `@path.txt` to execute SQL queries directly from a file:

In [6]:
with open('query_demo.txt', 'w') as f:
    f.write('select species, avg(body_mass_g) as avg_mass from data group by species order by avg_mass desc')

out = penguins.sql('@query_demo.txt')
if os.path.exists('query_demo.txt'):
    os.remove('query_demo.txt')
out

species,avg_mass
str,f64
"""Gentoo""",5076.01626
"""Chinstrap""",3733.088235
"""Adelie""",3700.662252


## Analytical SQL: Window Functions and Common Table Expressions (CTEs)

Polars SQLContext natively supports full analytical ANSI SQL features including window functions, partitioning, ranking, and common table expressions (CTEs):


In [7]:
query = """
WITH ranked AS (
    SELECT species, island, body_mass_g,
           RANK() OVER (PARTITION BY species ORDER BY body_mass_g DESC) as rank
    FROM data
    WHERE body_mass_g IS NOT NULL
)
SELECT * FROM ranked WHERE rank <= 2
"""
penguins.sql(query)


species,island,body_mass_g,rank
str,str,f64,i64
"""Adelie""","""Biscoe""",4725.0,2
"""Adelie""","""Biscoe""",4775.0,1
"""Chinstrap""","""Dream""",4550.0,2
"""Chinstrap""","""Dream""",4800.0,1
"""Gentoo""","""Biscoe""",6300.0,1
"""Gentoo""","""Biscoe""",6050.0,2
